In [4]:
import torch
import json
from types import SimpleNamespace
from torch.utils.data import DataLoader

from training.stage2_trainer_seamless import load_stage_1_checkpoint
from data_processing.seamless_dataloader import SeamlessDataLoader
from utils import rotation_conversions as rc
from models_dyadit.using_vqvae import get_codebook_representation

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load config mimicking train_stage2_trainer.py
with open("configs/stage2_model_local.json", "r") as f:
    config_dict = json.load(f)

args = SimpleNamespace(**config_dict)
args.data = SimpleNamespace(**config_dict["data"])
args.training_params = SimpleNamespace(**config_dict["training_params"])
args.stage_1_pretrain = SimpleNamespace(**config_dict["stage_1_pretrain"])

model_params, stage1_model = load_stage_1_checkpoint(
    args.stage_1_pretrain.checkpoint_path, 
    args.stage_1_pretrain.config_path, 
    device
)

dataset = SeamlessDataLoader(args.data.train_lmdb, stage=2)
dataloader = DataLoader(dataset, batch_size=args.training_params.batch_size, shuffle=False)


dict_data = next(iter(dataloader))
target_pose = dict_data["pose"].to(device, non_blocking=True)
batch_size, total_frames = target_pose.shape[0], target_pose.shape[1]


target_pose_matrix = rc.axis_angle_to_matrix(target_pose.reshape(batch_size, total_frames, 43, 3))
target_pose_6d = rc.matrix_to_rotation_6d(target_pose_matrix).reshape(batch_size, total_frames, 258)

# Extract tokens
with torch.no_grad():
    motion_codes = stage1_model.encode(target_pose_6d)

print(f"Raw Pose Shape: {target_pose.shape}")
print(f"Converted 6D Pose Shape: {target_pose_6d.shape}")
print(f"Extracted Motion Codes Shape (B, T, L): {motion_codes.shape}")
print(f"Motion codes shape for Frame 1: {motion_codes[:, 0, :].shape}")
print(f"Motion codes for Sample 0, Frame 0: {motion_codes[0, 0, :]}")

seen_shapes = set()
for dict_data in dataloader:
    audio_shape = tuple(dict_data["audio"].shape)
    if audio_shape not in seen_shapes:
        seen_shapes.add(audio_shape)
print(f"the following shapes are present for audio {seen_shapes}")

/home/david/thesis/MPI_Project/training/stage2_trainer_seamless.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_locatio

[[ 0.43828732 -0.06142597  0.08079009 ...  0.53305167 -0.38405168
   0.03614052]
 [ 0.43850422 -0.06134596  0.08067234 ...  0.5312893  -0.3836373
   0.0329328 ]
 [ 0.43865335 -0.06118542  0.08047066 ...  0.5311314  -0.38588765
   0.03077097]
 ...
 [ 0.45240837 -0.06178149  0.07650304 ...  0.46285662 -0.36754575
  -0.01668602]
 [ 0.450561   -0.06195378  0.07623875 ...  0.46546602 -0.36556908
  -0.01587886]
 [ 0.44970676 -0.0620218   0.07608631 ...  0.46616137 -0.36294684
  -0.01560147]]
[[ 0.44309863 -0.0022551  -0.00830567 ...  0.31255364 -0.25574803
   0.1944352 ]
 [ 0.44299483 -0.00235352 -0.00849438 ...  0.31652573 -0.25402778
   0.19976439]
 [ 0.44550753 -0.00254501 -0.00825701 ...  0.3153835  -0.2568175
   0.190087  ]
 ...
 [ 0.44822305 -0.01609719  0.08044361 ...  0.25023904 -0.24802859
   0.07965757]
 [ 0.44686985 -0.01610173  0.07907597 ...  0.262221   -0.24984759
   0.10412414]
 [ 0.44491756 -0.01601934  0.07768082 ...  0.26743487 -0.25255674
   0.1270916 ]]
[[ 0.4467868  -0.0

In [8]:
for key, value in dict_data.items():
    if hasattr(value, "shape"):
        print(f"{key} has a shape of: {value.shape}")

pose has a shape of: torch.Size([12, 64, 129])
audio has a shape of: torch.Size([12, 106, 1024])
bert has a shape of: torch.Size([12, 64, 768])
semantic_scores has a shape of: torch.Size([12, 64])
partner_audio has a shape of: torch.Size([12, 106, 1024])
partner_pose has a shape of: torch.Size([12, 64, 129])
partner_bert has a shape of: torch.Size([12, 64, 768])
conversation_states has a shape of: torch.Size([12, 64])
social_relationship has a shape of: torch.Size([12])
personality_traits has a shape of: torch.Size([12, 5])
emotions has a shape of: torch.Size([12, 64])


### Latent space

In [ ]:
print(motion_codes.shape)
latent_vectors = get_codebook_representation(stage1_model, motion_codes)
print(latent_vectors.shape)

torch.Size([32, 16, 4])
torch.Size([32, 16, 64])


In [ ]:
import torch.nn as nn
mse_loss_function = nn.MSELoss()

print(mse_loss_function(latent_vectors, latent_vectors).item())

0.0


### latent vector without helper function so it is differentiable

In [ ]:
print(stage1_model.quantizer.layers[0].codebook)
print(stage1_model.quantizer.layers[0].codebook.shape)

tensor([[-1.0407e-02,  7.7252e-02, -2.1352e-02,  ..., -1.8506e-01,
          1.3246e-01, -9.8445e-02],
        [-8.7685e-02,  2.6646e-03,  2.3591e-01,  ..., -2.2762e-02,
         -1.8812e-01, -7.6899e-02],
        [-1.4524e-01, -2.3103e-01, -9.1612e-02,  ...,  1.4290e-01,
          3.3520e-01, -3.0850e-02],
        ...,
        [-7.4009e-01, -7.3342e-02,  1.8560e-01,  ..., -3.4179e-01,
         -9.7608e-02, -5.4501e-03],
        [-2.8282e-01, -1.5527e-02, -1.3376e-02,  ..., -1.7113e-02,
         -1.0816e-01, -5.4207e-02],
        [-1.0324e-01, -1.8176e-01,  6.1723e-04,  ..., -1.3459e-02,
          3.4753e-02,  3.0740e-01]], device='cuda:0')
torch.Size([512, 64])


### Test masking

In [ ]:
import math
import random

def mask_ground_truth_tokens(max_mask_problem, step_percentage, input_tokens, code_dim):
    """Function to mask out ground truth tokens, to prevent overfitting and do masked learning"""
    cosine_mask_prob = max_mask_problem * (0.5 * (1.0 - math.cos(math.pi * step_percentage)))
    
    mask_token_id = code_dim # # Index code_dim is the 'MASK'
    token_mask_indices = torch.rand(input_tokens.shape) < cosine_mask_prob
    masked_tokens = input_tokens.clone()
    masked_tokens[token_mask_indices] = mask_token_id
    return masked_tokens

def mask_ground_truth_tokens_2(max_mask_prob, step_percentage, input_tokens, mask_token):
    """
    Masks the input tokens with the mask_token, will mask in blocks so not uniform random
    Will never mask the first motion code out of all layers, therefore the first head will always be trained with trainer enforcing
    """
    if max_mask_prob == 0:
        return input_tokens.clone()

    B, T, L = input_tokens.shape
    device = input_tokens.device
    cosine_mask_prob = max_mask_prob * (0.5 * (1.0 - math.cos(math.pi * step_percentage)))
    mask = torch.zeros((B, T, L), dtype=torch.bool, device=device)

    max_block_size = L-1 # Block masking, define the max block size
    block_size = max(1, math.ceil(max_block_size * step_percentage))
    num_blocks = max(1, int((T * max_block_size * cosine_mask_prob) / block_size)) # Calculate how many blocks to mask based on probability
    print(block_size, num_blocks)
    
    for b in range(B):
        for _ in range(num_blocks):
            start = random.randint(0, T - block_size)
            # Mask the chunk
            num_layers_to_mask = random.randint(1, L-1)
            random_layers = random.sample(range(1, L), num_layers_to_mask)
            mask[b, start : start + block_size, random_layers] = True
            
    # Force Layer 0 (Base) to always be visible to provide semantic stability
    mask[:, :, 0] = False 
    
    # Apply Mask
    masked_tokens = input_tokens.clone()
    masked_tokens[mask] = mask_token
    return masked_tokens


print(motion_codes.shape)
masked_motion_codes = mask_ground_truth_tokens_2(0.0, 1, motion_codes, 512)
print(masked_motion_codes.shape)
print(motion_codes[1])
print(masked_motion_codes[1])

torch.Size([32, 16, 4])
torch.Size([32, 16, 4])
tensor([[114, 282, 283, 316],
        [316, 393,  69, 360],
        [473, 123, 410, 392],
        [168, 274, 156, 237],
        [473, 154, 224, 438],
        [117, 487, 452,   9],
        [117, 487,  61,   9],
        [473, 339, 492, 486],
        [168, 120, 107, 401],
        [168, 120, 107, 237],
        [473, 249, 331, 218],
        [117, 487,  47, 298],
        [168, 120, 382, 132],
        [473, 249, 119, 218],
        [168, 120, 394,  26],
        [473, 123, 344,  20]], device='cuda:0')
tensor([[114, 282, 283, 316],
        [316, 393,  69, 360],
        [473, 123, 410, 392],
        [168, 274, 156, 237],
        [473, 154, 224, 438],
        [117, 487, 452,   9],
        [117, 487,  61,   9],
        [473, 339, 492, 486],
        [168, 120, 107, 401],
        [168, 120, 107, 237],
        [473, 249, 331, 218],
        [117, 487,  47, 298],
        [168, 120, 382, 132],
        [473, 249, 119, 218],
        [168, 120, 394,  26],
    